In [2]:
from pathlib import Path
import subprocess


repo_dir = Path.cwd() / "pinn_option_pricing"

# Clone the repository only if it has not already been downloaded
if not repo_dir.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/ashish1610dhiman/pinn_option_pricing.git"
        ],
        check=True
    )
    print("Repository downloaded successfully.")
else:
    print("Repository already exists.")

print("Repository location:")
print(repo_dir)

Repository already exists.
Repository location:
/Users/michellewan/PINN/pinn_option_pricing


In [3]:
import sys
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

european_call_dir = repo_dir / "european_call"

if str(european_call_dir) not in sys.path:
    sys.path.append(str(european_call_dir))


# Import the author's data class
from european_utils import EuropeanOptionData

In [4]:
device = torch.device("cpu")

In [5]:
# Use synthetic experiment parameter from the paper

K = 40.0
T = 1.0

t_min = 0.0     # today
t_max = T

S_min = 0.0
S_max = 160.0

# True used to generate synthetic data 作者设定的forward pricing experiment
r_true = 0.05
sigma_true = 0.20

# Create the author's European option data object
euro_call_data = EuropeanOptionData(
    t_range=[t_min, t_max],
    S_range=[S_min, S_max],
    K=K,
    r=r_true,
    sigma=sigma_true
)

In [6]:
# 测试作者的analytical solution
S_test = torch.tensor([[40.0]], dtype=torch.float32)
t_test = torch.tensor([[0.5]], dtype=torch.float32)

with torch.no_grad():
    V_test = euro_call_data.get_analytical_soln(
        S_test,
        t_test
    )

print("S =", S_test.item())
print("t =", t_test.item())
print("Analytical call price =", V_test.item())
print("Output shape =", V_test.shape)

S = 40.0
t = 0.5
Analytical call price = 2.755491256713867
Output shape = torch.Size([1, 1])


In [7]:
# Number of synthetic price observations
N_obs = 2000

epsilon = 1e-4

# Random interior observation locations
t_obs = (
    t_min + epsilon
    + (t_max - t_min - 2 * epsilon)
    * torch.rand(N_obs, 1)
)

S_obs = (
    S_min + epsilon
    + (S_max - S_min - 2 * epsilon)
    * torch.rand(N_obs, 1)
)

# Generate analytical Black-Scholes prices
with torch.no_grad():
    V_obs = euro_call_data.get_analytical_soln(
        S_obs,
        t_obs
    ).reshape(-1, 1)

# Convert to dataframe
synthetic_df = pd.DataFrame(
    {
        "t": t_obs.numpy().flatten(),
        "S": S_obs.numpy().flatten(),
        "V": V_obs.numpy().flatten()
    }
)

# Save as a NEW file
data_dir = Path("data")
data_dir.mkdir(parents=True, exist_ok=True)

data_path = (
    data_dir
    / "synthetic_european_call_data_2000.csv"
)

synthetic_df.to_csv(
    data_path,
    index=False
)

print("Dataset shape:", synthetic_df.shape)
print("Saved to:", data_path)

synthetic_df.head()

Dataset shape: (2000, 3)
Saved to: data/synthetic_european_call_data_2000.csv


,t,S,V
0,0.878904,65.789268,26.030731
1,0.443117,46.604435,8.051128
2,0.653018,66.463417,27.151398
3,0.534206,92.272247,53.193069
4,0.006199,114.922005,76.861038
